<a href="https://colab.research.google.com/github/AngelEstefano/Actividades-POO/blob/main/Proyecto_Final_(POO).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

INSTALACION DE STREAMLIT

In [321]:
!pip install -q streamlit

In [322]:
%%writefile app.py
import streamlit as st
import csv
import shutil
import os
import time
import smtplib
import random
from email.message import EmailMessage
from datetime import datetime
from IPython.display import clear_output

class Birthday:
  # Iniciador de la clase Cumpleaños
  def __init__(self, name, date, mail):
    self.name = name
    self.date = date
    self.mail = mail

  # Retorno de los valores en una cadena
  def __str__(self):
    return f"{self.name}, {self.date}, {self.mail}"

class BirthdayMenu:
  # Iniciador del archivo y del correo
  def __init__(self, birthday_reminder):
      # Crear una carpeta permanente para los datos
      self.data_dir = '/mnt/data/permanent_data'
      os.makedirs(self.data_dir, exist_ok=True)
      self.birthday_reminder = os.path.join(self.data_dir, f"{birthday_reminder}.csv")
      self.archive_csv()

  # Verifica si el archivo existe, si no, lo crea
  def archive_csv(self):
    if not os.path.exists(self.birthday_reminder):
            with open(self.birthday_reminder, mode='w', newline='') as file:
                writer = csv.writer(file)
                writer.writerow(['Name', 'Birthday', 'Mail'])

  # Funcion para registrar los cumpleaños
  def Birthday_Register(self, name, date, mail):
    with open(self.birthday_reminder, mode='a', newline='') as file:
      writer = csv.writer(file)
      writer.writerow([name, date, mail])

  def get_upcoming_birthdays(self):
        current_date = datetime.now().strftime("%d-%m")  # Fecha de hoy en formato dd-mm
        upcoming_birthdays = []

        with open(self.birthday_reminder, mode='r') as file:
            reader = csv.reader(file)
            next(reader, None)  # Saltar encabezado
            for row in reader:
                name, birthday, email = row
                if birthday.startswith(current_date):  # Verificar si el cumpleaños coincide con la fecha de hoy
                    upcoming_birthdays.append((name, birthday, email))
        return upcoming_birthdays

  # Funcion para obtener el correo asociado al nombre
  def get_email_by_name(self, name):
        with open(self.birthday_reminder, mode='r') as file:
            reader = csv.reader(file)
            next(reader, None)
            for row in reader:
                if row[0].strip().lower() == name.strip().lower():
                    return row[2]
        return None

# Ruta del archivo donde se guardarán los mensajes personalizados
CUSTOM_MESSAGES_FILE = "custom_messages.csv"

class BirthdayMessage:
    def __init__(self):
        # Inicializar el atributo para los mensajes generales
        self.general_message = ""  # Atributo para el mensaje general
        self.custom_messages = self.load_custom_messages()

    def load_custom_messages(self):
        """Carga los mensajes personalizados desde un archivo CSV."""
        if os.path.exists(CUSTOM_MESSAGES_FILE):
            with open(CUSTOM_MESSAGES_FILE, 'r') as file:
                reader = csv.reader(file)
                next(reader, None)  # Saltar encabezado si existe
                custom_messages = {}
                for row in reader:
                    name, message = row
                    if name not in custom_messages:
                        custom_messages[name] = []
                    custom_messages[name].append(message)
                return custom_messages
        else:
            return {}

    def save_custom_messages(self):
        """Guarda los mensajes personalizados en un archivo CSV."""
        with open(CUSTOM_MESSAGES_FILE, 'w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(['name', 'message'])  # Escribir encabezado
            for name, messages in self.custom_messages.items():
                for message in messages:
                    writer.writerow([name, message])

    def add_custom_message(self, name, message):
        """Agrega un mensaje personalizado para un nombre dado."""
        if name not in self.custom_messages:
            self.custom_messages[name] = []
        self.custom_messages[name].append(message)
        self.save_custom_messages()

    def list_custom_messages(self, name):
        """Devuelve la lista de mensajes personalizados para un nombre dado."""
        return self.custom_messages.get(name, [])

    def send_birthday_message(self, recipient_email, subject, message_body, recipient_name):
        message_body = message_body.replace("{name}", recipient_name)
        msg = EmailMessage()
        msg['Subject'] = subject
        msg['From'] = self.sender_email
        msg['To'] = recipient_email
        msg.set_content(message_body)

        try:
            with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
                server.login(self.sender_email, self.sender_password)
                server.send_message(msg)
            print(f"Mensaje enviado a {recipient_name} ({recipient_email})")
        except Exception as e:
            print(f"Error al enviar el mensaje: {e}")
############################################### PARTE FRONT-END ###############################################
birthday_menu = BirthdayMenu("birthday_reminder")

st.title("Aplicación de Envío y Recordatorio de Cumpleaños")

st.subheader("Cumpleaños Próximos")
def show_upcoming_birthdays():
    current_date = datetime.now()
    upcoming_birthdays = []
    with open(birthday_menu.birthday_reminder, mode='r') as file:
        reader = csv.reader(file)
        next(reader, None)
        for fila in reader:
            name, birthday, mail = fila
            birthdays = datetime.strptime(birthday.strip(), "%d-%m-%Y")
            birthdays_this_year = birthdays.replace(year=current_date.year)
            if birthdays_this_year < current_date:
                birthdays_this_year = birthdays_this_year.replace(year=current_date.year + 1)
            days_remaining = (birthdays_this_year - current_date).days
            upcoming_birthdays.append((name, birthday, days_remaining))
    upcoming_birthdays.sort(key=lambda x: x[2])
    if upcoming_birthdays:
        st.write("Los próximos cumpleaños son:")
        for name, birthday, days_remaining in upcoming_birthdays:
            st.write(f"{name}: {birthday} - Faltan {days_remaining} días")
    else:
        st.write("No hay cumpleaños próximos.")
show_upcoming_birthdays()

option = st.selectbox(
    "Elija una opción",
    ("Registrar Cumpleaños", "Configuración de Mensajes Generales", "Configuración de Mensajes Personalizados")
)

# Registrar cumpleaños
if option == "Registrar Cumpleaños":
    st.subheader("Registrar Nuevo Cumpleaños")
    name = st.text_input("Ingrese el nombre:")
    date = st.text_input("Ingrese la fecha de cumpleaños (dd-mm-aaaa):")
    mail = st.text_input("Ingrese el correo electrónico:")
    if st.button("Registrar"):
        birthday_menu.Birthday_Register(name, date, mail)
        st.success("Cumpleaños registrado correctamente")

# Configurar mensaje general
elif option == "Configuración de Mensajes Generales":
    st.subheader("Configurar Mensaje General")

    # Iniciar el mensaje general con correo y contraseña
    email = st.text_input("Ingrese su correo de envío:")
    password = st.text_input("Ingrese su contraseña de correo:", type="password")

    if 'birthday_message' not in st.session_state:
        st.session_state.birthday_message = BirthdayMessage()

    # Ingresar el mensaje general
    if 'birthday_message' in st.session_state:
        message = st.text_area("Ingrese el mensaje general para todos los cumpleaños:")

        # Guardar mensaje general y correo cuando el usuario lo confirme
        if st.button("Confirmar y Guardar Mensaje General"):
            if email and password and message:
                st.session_state.birthday_message.sender_email = email
                st.session_state.birthday_message.sender_password = password
                st.session_state.birthday_message.set_general_message(message)
                st.success("Mensaje general y correo configurados correctamente")
            else:
                st.warning("Por favor, ingrese un correo, contraseña y mensaje.")

        # Mostrar el mensaje que se ha configurado
        if st.session_state.birthday_message.general_message:
            st.write("Mensaje configurado: ", st.session_state.birthday_message.general_message)

    # **Enviar el mensaje general solo cuando se seleccione la opción "Mensajes Generales"**
    if st.button("Enviar Mensajes Generales"):
        if 'birthday_message' in st.session_state and st.session_state.birthday_message.general_message:
            st.subheader("Verificar y Enviar Mensajes Generales")
            upcoming_birthdays = birthday_menu.get_upcoming_birthdays()

            if upcoming_birthdays:
                for name, birthday, email in upcoming_birthdays:
                    # Enviar mensaje a los cumpleaños del día
                    st.session_state.birthday_message.send_birthday_message(
                        recipient_email=email,
                        subject="¡Feliz Cumpleaños!",
                        message_body=st.session_state.birthday_message.general_message,
                        recipient_name=name
                    )
                    st.write(f"Mensaje enviado a {name} en {birthday} a {email}")
            else:
                st.write("No hay cumpleaños hoy.")

# Inicializar 'birthday_message' en session_state si no está definido
if 'birthday_message' not in st.session_state:
    st.session_state.birthday_message = BirthdayMessage()

# Código de Streamlit para usar el mensaje personalizado
elif option == "Configuración de Mensajes Personalizados":
    st.subheader("Configurar Mensajes Personalizados")

    # Verificar que se ha configurado primero el correo y la contraseña de los mensajes generales
    if 'birthday_message' not in st.session_state:
        st.warning("Por favor, configure su correo de envío y contraseña en la sección de mensajes generales.")
    else:
        # Mostrar opciones de mensajes personalizados
        birthday_message = st.session_state.birthday_message
        st.write("Correo configurado correctamente para los mensajes generales.")

        # **Correo independiente para los mensajes personalizados**
        custom_email = st.text_input("Ingrese su correo de envío para mensajes personalizados:")
        custom_password = st.text_input("Ingrese su contraseña para mensajes personalizados:", type="password")

        if not custom_email or not custom_password:
            st.warning("Por favor, ingrese su correo y contraseña para enviar mensajes personalizados.")
        else:
            # Nombre para agregar mensaje personalizado
            name_for_custom_message = st.text_input("Ingrese el nombre para agregar mensaje personalizado:")

            if name_for_custom_message:
                action = st.selectbox("Seleccione una acción",
                                      ["Agregar mensaje personalizado",
                                       "Listar mensajes personalizados",
                                       "Seleccionar mensaje y enviar",
                                       "Enviar mensaje aleatorio"])

                if action == "Agregar mensaje personalizado":
                    custom_message = st.text_area("Escriba el mensaje personalizado:")
                    if st.button("Agregar"):
                        st.session_state.birthday_message.add_custom_message(name_for_custom_message, custom_message)
                        st.success(f"Mensaje personalizado agregado para {name_for_custom_message}")

                elif action == "Listar mensajes personalizados":
                    messages = st.session_state.birthday_message.list_custom_messages(name_for_custom_message)
                    if messages:
                        st.write(f"Mensajes personalizados para {name_for_custom_message}:")
                        for idx, msg in enumerate(messages):
                            st.write(f"{idx + 1}. {msg}")
                    else:
                        st.write(f"No hay mensajes personalizados para {name_for_custom_message}")

                elif action == "Seleccionar mensaje y enviar":
                    messages = st.session_state.birthday_message.list_custom_messages(name_for_custom_message)
                    if messages:
                        selected_message = st.selectbox("Seleccione un mensaje", messages)

                        # Mostrar el mensaje seleccionado
                        st.write(f"Mensaje seleccionado: {selected_message}")

                        if st.button("Confirmar selección"):
                            recipient_email = birthday_menu.get_email_by_name(name_for_custom_message)  # Se obtiene automáticamente el correo
                            if recipient_email:
                                st.session_state.birthday_message.sender_email = custom_email
                                st.session_state.birthday_message.sender_password = custom_password
                                st.session_state.birthday_message.send_birthday_message(recipient_email, "Cumpleaños", selected_message, name_for_custom_message)
                                st.success(f"Mensaje enviado a {name_for_custom_message} en {recipient_email}")
                            else:
                                st.warning("No se encontró el correo asociado con el nombre ingresado.")
                    else:
                        st.warning("No hay mensajes personalizados para seleccionar.")

                elif action == "Enviar mensaje aleatorio":
                    messages = st.session_state.birthday_message.list_custom_messages(name_for_custom_message)
                    recipient_email = birthday_menu.get_email_by_name(name_for_custom_message)  # Se obtiene automáticamente el correo
                    if recipient_email:
                        # Seleccionar un mensaje aleatorio si hay mensajes en la lista, o enviar uno predefinido
                        if messages:
                            random_message = random.choice(messages)
                        else:
                            random_message = "¡Feliz cumpleaños! Este es un mensaje predeterminado."

                        st.session_state.birthday_message.sender_email = custom_email
                        st.session_state.birthday_message.sender_password = custom_password
                        st.session_state.birthday_message.send_birthday_message(recipient_email, "Cumpleaños", random_message, name_for_custom_message)
                        st.success(f"Mensaje aleatorio enviado a {name_for_custom_message} en {recipient_email}")
                    else:
                        st.warning("No se encontró el correo asociado con el nombre ingresado.")

Overwriting app.py


In [323]:
!npm install localtunnel

⠙⠹⠸⠼⠴⠦
up to date, audited 23 packages in 2s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦
2 moderate severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠦

In [324]:
!streamlit run /content/app.py &>/content/logs.txt &

In [325]:
!npx localtunnel --port 8501

⠙your url is: https://nice-planets-sleep.loca.lt
^C
